# Đọc dữ liệu

In [20]:
# ==============================================================================
# CELL 1: IMPORT THƯ VIỆN VÀ ĐỌC DỮ LIỆU SẠCH (ĐÃ CHIA TRAIN/TEST TỪ TRƯỚC)
# ==============================================================================
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

# 1. Đọc thẳng 2 file Train và Test đã lưu ở bước Clean
df_train_clean = pd.read_csv('data_output/1_data_cleaned_train.csv')
df_test_clean = pd.read_csv('data_output/1_data_cleaned_test.csv')

print("✅ Đã tải dữ liệu Train và Test thành công!")

# 2. Tách lại Biến độc lập (X) và Biến mục tiêu (y)
# LƯU Ý: Chỉnh sửa 'Khoảng giá' thành tên cột mục tiêu chính xác của bạn nếu cần
target_col = 'Khoảng giá' 

X_train = df_train_clean.drop(columns=[target_col])
y_train = df_train_clean[target_col]

X_test = df_test_clean.drop(columns=[target_col])
y_test = df_test_clean[target_col]

print(f"Kích thước X_train: {X_train.shape}")
print(f"Kích thước X_test: {X_test.shape}")

✅ Đã tải dữ liệu Train và Test thành công!
Kích thước X_train: (4645, 25)
Kích thước X_test: (1164, 25)


In [21]:
print(X_train.columns)

Index(['Diện tích', 'Số phòng ngủ', 'Số phòng tắm, vệ sinh', 'Số tầng',
       'Đường vào', 'Pháp lý', 'Nội thất', 'Latitude', 'Longitude',
       'Quan_Huyen', 'Phuong_Xa', 'Dien_Tich_Lo', 'hem_xe_hoi',
       'gan_cho_sieu_thi', 'gan_truong_hoc', 'gan_benh_vien',
       'gan_cong_vien_ho_nuoc', 'duong_vao_null_flag', 'Duong',
       'Ty_Le_Giao_Thong', 'Ty_Le_Cong_Cong_CX', 'Ty_Le_Khac', 'Ty_Le_Dat_O',
       'Tổng số phòng', 'khoang_cach_trung_tam'],
      dtype='object')


In [22]:
print(X_train['Pháp lý'].unique())
print(X_train['Nội thất'].unique())

['Sổ hồng' 'Sổ đỏ' 'không có']
['Trống / Nhà thô' 'Cơ bản' 'Đầy đủ' 'Cao cấp']


# Encode các biến catergorical: 'Pháp lý', 'Nội thất'

In [23]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: ORDINAL ENCODING (PHÁP LÝ & NỘI THẤT)
# ═══════════════════════════════════════════════════════════════════════
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

# 1. Định nghĩa thứ tự cho từng cột (Từ thấp đến cao)
# Với Pháp lý: 'không có' (0) < 'Sổ đỏ' (1) < 'Sổ hồng' (2)
phap_ly_order = ['không có', 'Sổ đỏ', 'Sổ hồng']

# Với Nội thất: 'Trống / Nhà thô' (0) < 'Cơ bản' (1) < 'Đầy đủ' (2) < 'Cao cấp' (3)
noi_that_order = ['Trống / Nhà thô', 'Cơ bản', 'Đầy đủ', 'Cao cấp']

# 2. Khởi tạo bộ Ordinal Encoder
# - categories: Truyền danh sách các thứ tự đã định nghĩa ở trên (phải đúng thứ tự cột truyền vào)
# - handle_unknown='use_encoded_value' & unknown_value=-1: Nếu tập Test có nhãn lạ, gán bằng -1
orde = OrdinalEncoder(
    categories=[phap_ly_order, noi_that_order], 
    handle_unknown='use_encoded_value', 
    unknown_value=-1
)

# Các cột cần Ordinal Encoding
ord_cols = ['Pháp lý', 'Nội thất']

# 3. Học cấu trúc từ tập Train và biến đổi trực tiếp tập Train
# Khác với One-Hot sinh ra nhiều cột mới, Ordinal chỉ thay thế giá trị trên chính cột đó
X_train[ord_cols] = orde.fit_transform(X_train[ord_cols])

# 4. Chỉ biến đổi tập Test dựa trên tập luật đã học
X_test[ord_cols] = orde.transform(X_test[ord_cols])

print("✅ Đã áp dụng Ordinal Encoding thành công cho Pháp lý và Nội thất!")
print(f"Thứ tự Pháp lý đã mã hóa: {phap_ly_order} -> [0, 1, 2]")
print(f"Thứ tự Nội thất đã mã hóa: {noi_that_order} -> [0, 1, 2, 3]")

✅ Đã áp dụng Ordinal Encoding thành công cho Pháp lý và Nội thất!
Thứ tự Pháp lý đã mã hóa: ['không có', 'Sổ đỏ', 'Sổ hồng'] -> [0, 1, 2]
Thứ tự Nội thất đã mã hóa: ['Trống / Nhà thô', 'Cơ bản', 'Đầy đủ', 'Cao cấp'] -> [0, 1, 2, 3]


# Encode các biến về Địa chỉ. 

In [24]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: THỐNG KÊ SỐ LƯỢNG BẢN GHI THEO ĐƯỜNG
# ═══════════════════════════════════════════════════════════════════════

# 1. Đếm số lượng bản ghi theo cột 'Duong' và chuyển thành DataFrame cho đẹp
thong_ke_duong = X_train['Duong'].value_counts().reset_index()

# Đổi tên cột cho dễ đọc
thong_ke_duong.columns = ['Tên Đường', 'Số lượng bản ghi']

# 2. In ra 15 tuyến đường có nhiều bản ghi nhất
print("📍 TOP 15 ĐƯỜNG CÓ NHIỀU BẢN GHI NHẤT TRONG TẬP TRAIN:")
print(thong_ke_duong.head(15))

print("-" * 50)

# 3. (Tùy chọn) Kiểm tra xem có bao nhiêu tuyến đường hiếm (chỉ có 1-2 bản ghi)
# Điều này rất hữu ích để xác nhận lý do bạn cần dùng Smoothing cho Target Encoding
duong_hiem = thong_ke_duong[thong_ke_duong['Số lượng bản ghi'] <= 2]
print(f"⚠️ Số lượng tuyến đường chỉ xuất hiện 1-2 lần: {len(duong_hiem)} đường")

📍 TOP 15 ĐƯỜNG CÓ NHIỀU BẢN GHI NHẤT TRONG TẬP TRAIN:
                    Tên Đường  Số lượng bản ghi
0        Đường Huỳnh Tấn Phát               157
1           Đường Quang Trung                50
2         Đường Nơ Trang Long                44
3     Đường Xô Viết Nghệ Tĩnh                44
4             Đường Lê Văn Sỹ                43
5   Đường Cách Mạng Tháng Tám                40
6          Đường Lê Văn Lương                39
7         Đường Lạc Long Quân                37
8          Đường Phan Văn Trị                36
9       Đường Thích Quảng Đức                35
10       Đường Hoàng Hoa Thám                33
11          Đường Nguyễn Trãi                33
12        Đường Điện Biên Phủ                33
13         Đường Phan Huy Ích                32
14        Đường Lê Quang Định                31
--------------------------------------------------
⚠️ Số lượng tuyến đường chỉ xuất hiện 1-2 lần: 520 đường


In [25]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: TARGET ENCODING VỚI SMOOTHING (CẤU HÌNH RIÊNG CHO TỪNG CẤP ĐỘ)
# ═══════════════════════════════════════════════════════════════════════
import pandas as pd
from category_encoders import TargetEncoder

# -----------------------------------------------------------------------
# NHÓM 1: Cột 'Quận' (Dữ liệu lớn hơn -> Cần min_samples_leaf cao hơn)
# -----------------------------------------------------------------------
cols_quan = ['Quan_Huyen']
te_quan = TargetEncoder(cols=cols_quan, smoothing=10.0, min_samples_leaf=50)

# Học và biến đổi trên tập Train/Test
X_train[cols_quan] = te_quan.fit_transform(X_train[cols_quan], y_train)
X_test[cols_quan] = te_quan.transform(X_test[cols_quan])


# -----------------------------------------------------------------------
# NHÓM 2: Cột 'Phường/Xã' & 'Đường' (Dữ liệu nhỏ, phân tán -> min_samples_leaf thấp hơn)
# -----------------------------------------------------------------------
cols_phuong_duong = ['Phuong_Xa', 'Duong']
te_phuong_duong = TargetEncoder(cols=cols_phuong_duong, smoothing=10.0, min_samples_leaf=5)

# Học và biến đổi trên tập Train/Test
X_train[cols_phuong_duong] = te_phuong_duong.fit_transform(X_train[cols_phuong_duong], y_train)
X_test[cols_phuong_duong] = te_phuong_duong.transform(X_test[cols_phuong_duong])


print("✅ Đã áp dụng Target Encoding thành công!")
print("  - Cột 'Quận': min_samples_leaf = 50")
print("  - Cột 'Phường/Xã/Thị Trấn' & 'Đường_Từ_Địa_Chỉ': min_samples_leaf = 5")

✅ Đã áp dụng Target Encoding thành công!
  - Cột 'Quận': min_samples_leaf = 50
  - Cột 'Phường/Xã/Thị Trấn' & 'Đường_Từ_Địa_Chỉ': min_samples_leaf = 5


# Lưu dữ liệu

In [26]:
# ═══════════════════════════════════════════════════════════════════════
# CELL LƯU FILE 2: DỮ LIỆU DÀNH CHO MÔ HÌNH DẠNG CÂY (TREE-BASED MODELS)
# ═══════════════════════════════════════════════════════════════════════

# Tại đây, Pháp lý, Nội thất, Loại đường vào ĐÃ LÀ SỐ, nhưng Diện tích, Đường vào... CHƯA SCALE
# Tạo một bản sao sâu (deep copy) để lưu lại trạng thái này trước khi bị gán đè scale ở các cell sau
X_train_tree = X_train.copy()
X_test_tree = X_test.copy()

# Ghép với target gốc (Mô hình cây chạy tốt trên cả target gốc lẫn target log)  
df_train_tree = pd.concat([X_train_tree, y_train], axis=1)
df_test_tree = pd.concat([X_test_tree, y_test], axis=1) if 'y_test' in locals() else X_test_tree.copy()

# Xuất file
df_train_tree.to_csv('data_output/2_data_preprocessed_tree_train.csv', index=False)
df_test_tree.to_csv('data_output/2_data_preprocessed_tree_test.csv', index=False)

print("💾 FILE 2: Đã lưu dữ liệu cho mô hình dạng Cây thành công!")
print("   -> data_output/2_data_preprocessed_tree_train.csv")
print("   -> data_output/2_data_preprocessed_tree_test.csv")

💾 FILE 2: Đã lưu dữ liệu cho mô hình dạng Cây thành công!
   -> data_output/2_data_preprocessed_tree_train.csv
   -> data_output/2_data_preprocessed_tree_test.csv


In [27]:
print(X_train.columns)

Index(['Diện tích', 'Số phòng ngủ', 'Số phòng tắm, vệ sinh', 'Số tầng',
       'Đường vào', 'Pháp lý', 'Nội thất', 'Latitude', 'Longitude',
       'Quan_Huyen', 'Phuong_Xa', 'Dien_Tich_Lo', 'hem_xe_hoi',
       'gan_cho_sieu_thi', 'gan_truong_hoc', 'gan_benh_vien',
       'gan_cong_vien_ho_nuoc', 'duong_vao_null_flag', 'Duong',
       'Ty_Le_Giao_Thong', 'Ty_Le_Cong_Cong_CX', 'Ty_Le_Khac', 'Ty_Le_Dat_O',
       'Tổng số phòng', 'khoang_cach_trung_tam'],
      dtype='object')
